In [2]:

import os
import requests

if not os.path.exists("the-verdict.txt"):
    url = (
        "https://raw.githubusercontent.com/rasbt/"
        "LLMs-from-scratch/main/ch02/01_main-chapter-code/"
        "the-verdict.txt"
    )
    file_path = "the-verdict.txt"

    response = requests.get(url, timeout=30)
    response.raise_for_status()
    with open(file_path, "wb") as f:
        f.write(response.content)

In [3]:
with open("the-verdict.txt", "r", encoding = "utf-8") as f:
    raw_text = f.read()
print("Total no of characters:", len(raw_text))
print(raw_text[:99])

Total no of characters: 20479
I HAD always thought Jack Gisburn rather a cheap genius--though a good fellow enough--so it was no 


In [4]:
import re
text = "Hello, world. This is a test."
result = re.split(r'(\s)', text)
print(result)

['Hello,', ' ', 'world.', ' ', 'This', ' ', 'is', ' ', 'a', ' ', 'test.']


In [9]:
result = re.split(r'([,.]|\s)', text)
print(result)

['Hello', ',', '', ' ', 'world', '.', '', ' ', 'This', ' ', 'is', ' ', 'a', ' ', 'test', '.', '']


In [12]:
result = [ item for item in result if item.strip()]
print(result)
#removing space

['Hello', ',', 'world', '.', 'This', 'is', 'a', 'test', '.']


In [13]:
#modifying the scheme
text = "Hello, world. Is this-- a test?"
result = re.split(r'([,.:;?_!"()\']|--|\s)', text)
result = [item.strip() for item in result if item.strip()]
print(result)

['Hello', ',', 'world', '.', 'Is', 'this', '--', 'a', 'test', '?']


In [18]:
preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', raw_text)
preprocessed = [item.strip() for item in preprocessed if item.strip()]
print(preprocessed[30::-1])

['the', 'in', ',', 'that', 'hear', 'to', 'me', 'to', 'surprise', 'great', 'no', 'was', 'it', 'so', '--', 'enough', 'fellow', 'good', 'a', 'though', '--', 'genius', 'cheap', 'a', 'rather', 'Gisburn', 'Jack', 'thought', 'always', 'HAD', 'I']


In [20]:
all_words = sorted(set(preprocessed))
vocab_size = len(all_words)
print(vocab_size)

1130


In [21]:
vocab = {token: integer for integer, token in enumerate(all_words)}
for i, item in enumerate(vocab.items()):
    print(item)
    if i >= 50:
        break

('!', 0)
('"', 1)
("'", 2)
('(', 3)
(')', 4)
(',', 5)
('--', 6)
('.', 7)
(':', 8)
(';', 9)
('?', 10)
('A', 11)
('Ah', 12)
('Among', 13)
('And', 14)
('Are', 15)
('Arrt', 16)
('As', 17)
('At', 18)
('Be', 19)
('Begin', 20)
('Burlington', 21)
('But', 22)
('By', 23)
('Carlo', 24)
('Chicago', 25)
('Claude', 26)
('Come', 27)
('Croft', 28)
('Destroyed', 29)
('Devonshire', 30)
('Don', 31)
('Dubarry', 32)
('Emperors', 33)
('Florence', 34)
('For', 35)
('Gallery', 36)
('Gideon', 37)
('Gisburn', 38)
('Gisburns', 39)
('Grafton', 40)
('Greek', 41)
('Grindle', 42)
('Grindles', 43)
('HAD', 44)
('Had', 45)
('Hang', 46)
('Has', 47)
('He', 48)
('Her', 49)
('Hermia', 50)


### Our Tokenizer Structure

Our tokenizer will contain two core methods:
* **`encode`**: Takes the input text and converts it into a list of token IDs (`token_id`).
* **`decode`**: Takes the token IDs generated by the LLM and converts them back into a human-readable text string.

#### Class Attributes
To make our data processing pipeline smooth and efficient, we will store the following attributes:
* **`str_to_int`**: The vocabulary dictionary (`vocab`) stored as a class variable/attribute mapping text tokens to integer IDs.
* **`int_to_str`**: The inverse lookup mapping (`id: token`) to decode integers back to text tokens.


In [29]:
#making a whole class of tokenizer

class SimpleTokenizerV1:
    def __init__(self, vocab):
        self.str_to_int = vocab
        self.int_to_str = {i:s for s,i in vocab.items()}
    
    def encode(self, text):
        preprocessed = re.split(r'([,.?_!"()\']|--|\s)', text) # text -> tokens
        preprocessed = [item.strip() for item in preprocessed if item.strip()] # removes extra whitespaces
        ids = [self.str_to_int[s] for s in preprocessed] # given token_list to token_id list
        return ids
    
    def decode(self, ids):
        text = " ".join(self.int_to_str[i] for i in ids) #token_ids -> token + spaces in b/w
        text = re.sub(r'\s+([,.?!"()\'])', r'\1', text) #removes space before specified punctuation
        return text

In [37]:
tokenizer = SimpleTokenizerV1(vocab)
text = """"It's the last he painted, you know"
        Mrs. Gisburn said with pardonable pride."""
ids = tokenizer.encode(text)
print(ids)


[1, 56, 2, 850, 988, 602, 533, 746, 5, 1126, 596, 1, 67, 7, 38, 851, 1108, 754, 793, 7]


In [38]:
print(tokenizer.decode(ids))

" It' s the last he painted, you know" Mrs. Gisburn said with pardonable pride.


#### Remember this encode/decode method only works on tokens which are present in the vocabulary
They will not work on extra texts...
It will throw a `KeyError: 'extra_text'`

* So it's advised to take our vocabulary very large and diverse when working on LLMs



In [39]:
print(tokenizer.encode('Hello'))

KeyError: 'Hello'

#### Solution is to have some special extra context tokens in vocab
 eg. `<|unk|>` , `<|endoftext|>`

 `unk` is used for unknown texts<br>
 `endoftext` is used to separate texts from different sources and unrelated things


In [40]:
all_tokens = sorted(list(set(preprocessed)))
all_tokens.extend(["<|endoftext|>", "<|unk|>"])
vocab = {token: integer for integer, token in enumerate(all_tokens)}

print(len(vocab))

1132


In [41]:
for item in enumerate(list(vocab.items())[-5:]):
    print(item)

(0, ('younger', 1127))
(1, ('your', 1128))
(2, ('yourself', 1129))
(3, ('<|endoftext|>', 1130))
(4, ('<|unk|>', 1131))


In [42]:
#making a whole class of tokenizer

class SimpleTokenizerV2:
    def __init__(self, vocab):
        self.str_to_int = vocab
        self.int_to_str = {i:s for s,i in vocab.items()}
    
    def encode(self, text):
        preprocessed = re.split(r'([,.?_!"()\']|--|\s)', text) # text -> tokens
        preprocessed = [item.strip() for item in preprocessed if item.strip()] # removes extra whitespaces
        preprocessed = [item if item in self.str_to_int else "<|unk|>" for item in preprocessed]
        # it will replace unknown text to <|unk|>
        ids = [self.str_to_int[s] for s in preprocessed] # given token_list to token_id list
        return ids
    
    def decode(self, ids):
        text = " ".join(self.int_to_str[i] for i in ids) #token_ids -> token + spaces in b/w
        text = re.sub(r'\s+([,.?!"()\'])', r'\1', text) #removes space before specified punctuation
        return text

In [47]:
text1 = "Hello, I am Kunal"
text2 = "Bangalore is a cool city"
text = " <|endoftext|> ".join((text1,text2))
print(text)

Hello, I am Kunal <|endoftext|> Bangalore is a cool city


##### Now if we try to encode this using our tokenizer, it will not throw a KeyError

In [51]:
tokenizer = SimpleTokenizerV2(vocab)
ids = tokenizer.encode(text)
print(ids)

[1131, 5, 53, 150, 1131, 1130, 1131, 584, 115, 1131, 1131]


In [52]:
print(tokenizer.decode(ids))

<|unk|>, I am <|unk|> <|endoftext|> <|unk|> is a <|unk|> <|unk|>


In [3]:
from importlib.metadata import version
import tiktoken
print("tiktoken version:", version("tiktoken"))

tiktoken version: 0.13.0


#### Using pre-built Byte-Pair Encoding Tokenizer which is advanced in terms of capabilities

* The BPE tokenizer encodes and decoded unknown words, such as someUnknownPlace, correctly<br>
* The BPE tokenizer can handle any unknown word
* The algorithm underlying BPE breaks down words that aren't in its pre-defined vocab into smaller subword units or even characters,enabling it handle OOV words

In [4]:
tokenizer = tiktoken.get_encoding("gpt2")

In [7]:
text = "Hello this is Kunal!! <|endoftext|> NY is a good city"
integers = tokenizer.encode(text, allowed_special={"<|endoftext|>"})
print(integers)

[15496, 428, 318, 509, 18835, 3228, 220, 50256, 6645, 318, 257, 922, 1748]


In [8]:
strings = tokenizer.decode(integers)
print(strings)

Hello this is Kunal!! <|endoftext|> NY is a good city


In [9]:
sample_ex = "Akwirw ier"
sample_int = tokenizer.encode(sample_ex)
print(sample_int)

[33901, 86, 343, 86, 220, 959]


In [15]:
for i in range(len(sample_int)):
    sample_str = tokenizer.decode([sample_int[i]])
    print(sample_str)
    


Ak
w
ir
w
 
ier


In [16]:
with open("the-verdict.txt", "r",encoding="utf-8") as f:
    raw_text = f.read()

enc_text = tokenizer.encode(raw_text)
print(len(enc_text))

5145


In [19]:
enc_sample = enc_text[50:]
#removing some words for demonstration purpose

#### Creating input target pair

In [20]:
context_size = 4
x = enc_sample[:context_size]
y = enc_sample[1:context_size + 1]
print(f"x: {x}")
print(f"y:      {y}")

x: [290, 4920, 2241, 287]
y:      [4920, 2241, 287, 257]


In [21]:
for i in range(1, context_size +1):
    context = enc_sample[:i]
    desired = enc_sample[i]
    print(context, "---->", desired)

[290] ----> 4920
[290, 4920] ----> 2241
[290, 4920, 2241] ----> 287
[290, 4920, 2241, 287] ----> 257


In [25]:
for i in range(1, context_size +1):
    context = enc_sample[:i]
    desired = enc_sample[i]
    print(tokenizer.decode(context), "---->", tokenizer.decode([desired]))

 and ---->  established
 and established ---->  himself
 and established himself ---->  in
 and established himself in ---->  a


In [29]:
import torch
from torch.utils.data import Dataset, DataLoader

class GPTDatasetV1(Dataset):
    def __init__(self, txt, tokenizer, max_length, stride):
        self.input_ids = []
        self.target_ids = []
        
        token_ids = tokenizer.encode(txt)
        
        
        for i in range(0, len(token_ids) - max_length, stride):
            input_chunk = token_ids[i:i + max_length]
            target_chunk = token_ids[i+1:i + max_length + 1]
            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))
    
    def __len__(self):
        return len(self.input_ids)
    
    def __getitem__(self, idx):
        return self.input_ids[idx], self.target_ids[idx]
            

In [32]:
def create_dataloader_v1(txt, batch_size = 4, max_length = 256, stride = 128,
                         shuffle = True, drop_last = True, num_workers = 0):
    tokenizer = tiktoken.get_encoding("gpt2")
    dataset = GPTDatasetV1(txt, tokenizer, max_length, stride)
    dataloader = DataLoader(
        dataset,
        batch_size = batch_size,
        shuffle = shuffle,
        drop_last = drop_last,  #drops the last batch if its shorter than specified batch size
        num_workers = num_workers # no of CPU processes to use for preprocessing
    )
    
    return dataloader

In [41]:
#testing the dataloader with batch size of 1 ...
with open("the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

dataloader = create_dataloader_v1(
    raw_text, batch_size = 1, max_length = 4, stride = 1, shuffle = False
)

data_iter = iter(dataloader) #converts dataloader into Python iterator to fetch next entry by next()
first_batch = next(data_iter)

print(first_batch)

[tensor([[  40,  367, 2885, 1464]]), tensor([[ 367, 2885, 1464, 1807]])]


In [42]:
second_batch = next(data_iter)
print(second_batch)

#stride value dictates how many position we move after a batch

[tensor([[ 367, 2885, 1464, 1807]]), tensor([[2885, 1464, 1807, 3619]])]


In [43]:
dataloader = create_dataloader_v1(
    raw_text,
    batch_size = 8,
    max_length = 4,
    stride = 4,
    shuffle = False
)

In [44]:
data_iter = iter(dataloader)
inputs, targets = next(data_iter)
print(f"Inputs: \n {inputs}")
print(f"Targets: \n {targets}")

Inputs: 
 tensor([[   40,   367,  2885,  1464],
        [ 1807,  3619,   402,   271],
        [10899,  2138,   257,  7026],
        [15632,   438,  2016,   257],
        [  922,  5891,  1576,   438],
        [  568,   340,   373,   645],
        [ 1049,  5975,   284,   502],
        [  284,  3285,   326,    11]])
Targets: 
 tensor([[  367,  2885,  1464,  1807],
        [ 3619,   402,   271, 10899],
        [ 2138,   257,  7026, 15632],
        [  438,  2016,   257,   922],
        [ 5891,  1576,   438,   568],
        [  340,   373,   645,  1049],
        [ 5975,   284,   502,   284],
        [ 3285,   326,    11,   287]])


### Now moving to create Token Embeddings

In [52]:
vocab_size = 50257
output_dim = 256
max_length = 4
token_embedding_layer = torch.nn.Embedding(vocab_size, output_dim)
token_embeddings = token_embedding_layer(inputs)
print(f"Shape -> {token_embeddings.shape}")

Shape -> torch.Size([8, 4, 256])


[8,4,256] means In a batch of 8 we have 4 texts in each batch and each text has 256 dim vector embedding

In [55]:
#Positional encoding

context_length = max_length
pos_embedding_layer = torch.nn.Embedding(context_length, output_dim)
pos_embeddings = pos_embedding_layer(torch.arange(context_length))
print(pos_embeddings.shape)

torch.Size([4, 256])


In [57]:
input_embeddings = token_embeddings + pos_embeddings
print(input_embeddings.shape)

#by default Pytorch adds the position embedding to each batch

torch.Size([8, 4, 256])
